# 🔬 OptoBind-AI — Pipeline de Design De Novo de Biosenseurs Optiques

Ce notebook exécute **OptoBind-AI** dans un environnement propre **Python 3.10 + CUDA** (via Micromamba) sur un **GPU NVIDIA gratuit dans Google Colab**, garantissant 100% de compatibilité avec RFdiffusion All-Atom et ProteinMPNN.

---

### Étape 1 : Vérification du GPU NVIDIA T4
*(Assurez-vous que le GPU est actif dans Exécution > Modifier le type d'exécution > T4 GPU)*

In [ ]:
!nvidia-smi

### Étape 2 : Cloner le projet Optobind-AI

In [ ]:
%cd /content
!rm -rf Optobind-AI
!git clone https://github.com/Lorenzo-colao/Optobind-AI.git
%cd /content/Optobind-AI

!git clone https://github.com/RosettaCommons/RFdiffusion.git 2>/dev/null || true
!git clone https://github.com/dauparas/ProteinMPNN.git 2>/dev/null || true

print("\n✓ Dépôts sources clonés !")

### Étape 3 : Installation de l'environnement Python 3.10 (Compatible RFdiffusion & DGL)

In [ ]:
%%time
import os

# 1. Télécharger Micromamba (gestionnaire d'environnement rapide)
!wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba > /dev/null

# 2. Créer l'environnement Python 3.10 avec PyTorch et DGL compatibles CUDA
!./bin/micromamba create -y -n optobind -c pytorch -c nvidia -c dglteam -c conda-forge \
    python=3.10 \
    pytorch=2.1.2 \
    pytorch-cuda=11.8 \
    dgl-cuda11.8=1.1.2 \
    rdkit \
    biopython \
    openpyxl \
    pyyaml \
    pandas \
    scipy \
    tqdm

# 3. Installer les dépendances spécifiques dans cet environnement Python 3.10
!./bin/micromamba run -n optobind pip install --quiet hydra-core omegaconf pynvml decorator e3nn==0.3.3 chai-lab fair-esm
!./bin/micromamba run -n optobind pip install --quiet git+https://github.com/NVIDIA/dllogger#egg=dllogger

# 4. Installer SE3Transformer et RFdiffusion
!./bin/micromamba run -n optobind bash -c 'cd RFdiffusion/env/SE3Transformer && pip install --quiet .'
!./bin/micromamba run -n optobind bash -c 'cd RFdiffusion && pip install --quiet --no-deps -e .'

# 5. Télécharger les poids des modèles s'ils ne sont pas déjà présents
!mkdir -p weights
!wget -c -P weights/ http://files.ipd.uw.edu/pub/RF-All-Atom/weights/RFDiffusionAA_paper_weights.pt
!wget -c -O weights/ProteinMPNN_v_48_020.pt https://raw.githubusercontent.com/dauparas/ProteinMPNN/main/vanilla_model_weights/v_48_020.pt

print("\n🎉 ENVIRONNEMENT PYTHON 3.10 + CUDA TOTALEMENT OPÉRATIONNEL !")

### Étape 4 : Lancer le pipeline OptoBind-AI sur GPU !

In [ ]:
!./bin/micromamba run -n optobind python orchestrator.py \
  --target inputs/targets/zn2+_site.pdb \
  --ligand inputs/ligands/DFHBI-1T.sdf

### Étape 5 : Télécharger votre rapport Excel de validation

In [ ]:
import glob
from google.colab import files

# Recherche du rapport généré
reports = glob.glob('outputs/**/reports/report.xlsx', recursive=True)
if reports:
    print(f"Téléchargement de : {reports[-1]}")
    files.download(reports[-1])
else:
    print("Aucun rapport trouvé.")